# Day 8 v3 — Silver Layer: All 17 Entities (Production + Data Quality)
**Bronze Volume JSON → Silver Delta Tables**

**Source:** `/Volumes/dbw_ev_intelligence_dev/default/bronze-volume/api/<entity>/`
**Sink:** `/Volumes/dbw_ev_intelligence_dev/default/silver-volume/api/<entity>/` (Delta)

**What v3 adds over v2 (for loop version):**

| Scenario | What it does |
|---|---|
| Null primary key | Drop rows where `natural_key` is NULL — cannot identify the record |
| Duplicate primary key | Keep only the latest version per key (`updated_at desc`) |
| Corrupt / unparseable data | Rows that fail type casting produce NULL — quarantined to a reject table |
| Null `updated_at` | Drop rows with no CDC timestamp — cannot determine record freshness |
| String whitespace | Trim leading/trailing spaces on all string columns |
| Negative numeric values | Flag rows with negative amounts/prices — written to quarantine, not Silver |
| Full vs incremental | Full = overwrite, Incremental = Delta MERGE upsert on natural key |
| Quarantine table | Bad rows saved separately to `/silver-volume/quarantine/<entity>/` for investigation |

In [ ]:
# Cell 1 — Imports
from pyspark.sql import functions as F
from pyspark.sql.types import MapType, StringType
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from datetime import datetime, timezone

print("Imports OK")

In [ ]:
# Cell 2 — Widget: load_type
dbutils.widgets.removeAll()
dbutils.widgets.text("load_type", "incremental", "Load Type (full / incremental)")
dbutils.widgets.text("ingestion_date", "", "Ingestion Date (YYYY-MM-DD, blank = latest)")

LOAD_TYPE      = dbutils.widgets.get("load_type").strip().lower()
INGESTION_DATE = dbutils.widgets.get("ingestion_date").strip()

if LOAD_TYPE not in ("full", "incremental"):
    raise ValueError(f"load_type must be 'full' or 'incremental', got: {LOAD_TYPE!r}")

print(f"load_type      : {LOAD_TYPE}")
print(f"ingestion_date : {INGESTION_DATE or '(latest available)'}")

In [ ]:
# Cell 3 — Constants
BRONZE_VOLUME = "/Volumes/dbw_ev_intelligence_dev/default/bronze-volume"
SILVER_VOLUME = "/Volumes/dbw_ev_intelligence_dev/default/silver-volume"

BRONZE_API_BASE    = f"{BRONZE_VOLUME}/api"
SILVER_API_BASE    = f"{SILVER_VOLUME}/api"
QUARANTINE_BASE    = f"{SILVER_VOLUME}/quarantine"   # bad rows land here

PIPELINE_NAME = "pl_silver_api_transformation_v3"
RUN_TS        = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

print(f"BRONZE_API_BASE  : {BRONZE_API_BASE}")
print(f"SILVER_API_BASE  : {SILVER_API_BASE}")
print(f"QUARANTINE_BASE  : {QUARANTINE_BASE}")
print(f"RUN_TS           : {RUN_TS}")

In [ ]:
# Cell 4 — Entity schema registry
# Each entry: (natural_key, cdc_field, schema_dict)
# schema_dict maps column_name -> spark type string used in CAST

ENTITY_REGISTRY = {
    "payments": {
        "natural_key" : "payment_id",
        "cdc_field"   : "updated_at",
        "cast": {
            "payment_id"    : "string",
            "session_id"    : "string",
            "customer_id"   : "string",
            "amount"        : "decimal(10,2)",
            "currency"      : "string",
            "status"        : "string",
            "payment_method": "string",
            "created_at"    : "timestamp",
            "updated_at"    : "timestamp",
        }
    },
    "sessions": {
        "natural_key" : "session_id",
        "cdc_field"   : "updated_at",
        "cast": {
            "session_id"       : "string",
            "vehicle_id"       : "string",
            "charger_id"       : "string",
            "customer_id"      : "string",
            "station_id"       : "string",
            "start_time"       : "timestamp",
            "end_time"         : "timestamp",
            "energy_kwh"       : "decimal(10,4)",
            "duration_minutes" : "integer",
            "status"           : "string",
            "created_at"       : "timestamp",
            "updated_at"       : "timestamp",
        }
    },
    "customers": {
        "natural_key" : "customer_id",
        "cdc_field"   : "updated_at",
        "cast": {
            "customer_id"    : "string",
            "first_name"     : "string",
            "last_name"      : "string",
            "email"          : "string",
            "phone"          : "string",
            "city"           : "string",
            "state"          : "string",
            "country"        : "string",
            "created_at"     : "timestamp",
            "updated_at"     : "timestamp",
        }
    },
    "fleet": {
        "natural_key" : "vehicle_id",
        "cdc_field"   : "updated_at",
        "cast": {
            "vehicle_id"        : "string",
            "make"              : "string",
            "model"             : "string",
            "year"              : "integer",
            "battery_capacity"  : "decimal(8,2)",
            "range_km"          : "decimal(8,2)",
            "status"            : "string",
            "partner_id"        : "string",
            "created_at"        : "timestamp",
            "updated_at"        : "timestamp",
        }
    },
    "chargers": {
        "natural_key" : "charger_id",
        "cdc_field"   : "updated_at",
        "cast": {
            "charger_id"     : "string",
            "station_id"     : "string",
            "charger_type"   : "string",
            "power_kw"       : "decimal(8,2)",
            "status"         : "string",
            "connector_type" : "string",
            "created_at"     : "timestamp",
            "updated_at"     : "timestamp",
        }
    },
    "vehicles": {
        "natural_key" : "vehicle_id",
        "cdc_field"   : "updated_at",
        "cast": {
            "vehicle_id"       : "string",
            "customer_id"      : "string",
            "make"             : "string",
            "model"            : "string",
            "year"             : "integer",
            "battery_capacity" : "decimal(8,2)",
            "range_km"         : "decimal(8,2)",
            "license_plate"    : "string",
            "created_at"       : "timestamp",
            "updated_at"       : "timestamp",
        }
    },
    "stations": {
        "natural_key" : "station_id",
        "cdc_field"   : "updated_at",
        "cast": {
            "station_id"   : "string",
            "name"         : "string",
            "city"         : "string",
            "state"        : "string",
            "country"      : "string",
            "latitude"     : "decimal(10,7)",
            "longitude"    : "decimal(10,7)",
            "total_chargers": "integer",
            "status"       : "string",
            "created_at"   : "timestamp",
            "updated_at"   : "timestamp",
        }
    },
    "complaints": {
        "natural_key" : "complaint_id",
        "cdc_field"   : "updated_at",
        "cast": {
            "complaint_id"   : "string",
            "customer_id"    : "string",
            "session_id"     : "string",
            "category"       : "string",
            "description"    : "string",
            "status"         : "string",
            "priority"       : "string",
            "created_at"     : "timestamp",
            "updated_at"     : "timestamp",
        }
    },
    "maintenance_events": {
        "natural_key" : "event_id",
        "cdc_field"   : "updated_at",
        "cast": {
            "event_id"       : "string",
            "charger_id"     : "string",
            "station_id"     : "string",
            "event_type"     : "string",
            "description"    : "string",
            "technician_id"  : "string",
            "status"         : "string",
            "scheduled_date" : "date",
            "completed_date" : "date",
            "created_at"     : "timestamp",
            "updated_at"     : "timestamp",
        }
    },
    "energy_prices": {
        "natural_key" : "price_id",
        "cdc_field"   : "updated_at",
        "cast": {
            "price_id"       : "string",
            "region"         : "string",
            "price_per_kwh"  : "decimal(8,4)",
            "currency"       : "string",
            "effective_from" : "timestamp",
            "effective_to"   : "timestamp",
            "created_at"     : "timestamp",
            "updated_at"     : "timestamp",
        }
    },
    "tariffs": {
        "natural_key" : "tariff_id",
        "cdc_field"   : "updated_at",
        "cast": {
            "tariff_id"      : "string",
            "name"           : "string",
            "price_per_kwh"  : "decimal(8,4)",
            "price_per_min"  : "decimal(8,4)",
            "currency"       : "string",
            "valid_from"     : "timestamp",
            "valid_to"       : "timestamp",
            "created_at"     : "timestamp",
            "updated_at"     : "timestamp",
        }
    },
    "charge_cards": {
        "natural_key" : "card_id",
        "cdc_field"   : "updated_at",
        "cast": {
            "card_id"        : "string",
            "customer_id"    : "string",
            "card_number"    : "string",
            "status"         : "string",
            "issued_at"      : "timestamp",
            "expires_at"     : "timestamp",
            "created_at"     : "timestamp",
            "updated_at"     : "timestamp",
        }
    },
    "employees": {
        "natural_key" : "employee_id",
        "cdc_field"   : "updated_at",
        "cast": {
            "employee_id"  : "string",
            "first_name"   : "string",
            "last_name"    : "string",
            "email"        : "string",
            "role"         : "string",
            "department"   : "string",
            "station_id"   : "string",
            "hire_date"    : "date",
            "created_at"   : "timestamp",
            "updated_at"   : "timestamp",
        }
    },
    "partners": {
        "natural_key" : "partner_id",
        "cdc_field"   : "updated_at",
        "cast": {
            "partner_id"   : "string",
            "name"         : "string",
            "country"      : "string",
            "contact_email": "string",
            "status"       : "string",
            "created_at"   : "timestamp",
            "updated_at"   : "timestamp",
        }
    },
    "cities": {
        "natural_key" : "city_id",
        "cdc_field"   : "updated_at",
        "cast": {
            "city_id"      : "string",
            "name"         : "string",
            "state_code"   : "string",
            "country"      : "string",
            "latitude"     : "decimal(10,7)",
            "longitude"    : "decimal(10,7)",
            "population"   : "long",
            "created_at"   : "timestamp",
            "updated_at"   : "timestamp",
        }
    },
    "states": {
        "natural_key" : "state_code",
        "cdc_field"   : "updated_at",
        "cast": {
            "state_code"   : "string",
            "name"         : "string",
            "country"      : "string",
            "created_at"   : "timestamp",
            "updated_at"   : "timestamp",
        }
    },
    "weather": {
        "natural_key" : "city_id",
        "cdc_field"   : "updated_at",
        "cast": {
            "city_id"          : "string",
            "temperature_c"    : "decimal(5,2)",
            "humidity_pct"     : "decimal(5,2)",
            "wind_speed_kmh"   : "decimal(6,2)",
            "condition"        : "string",
            "recorded_at"      : "timestamp",
            "created_at"       : "timestamp",
            "updated_at"       : "timestamp",
        }
    },
}

print(f"Entity registry loaded: {len(ENTITY_REGISTRY)} entities")

In [ ]:
# Cell 5 — Helper functions

def get_bronze_paths(entity_name, ingestion_date):
    base = f"{BRONZE_API_BASE}/{entity_name}"
    try:
        dbutils.fs.ls(base)
    except Exception:
        return []
    if ingestion_date:
        try:
            files = dbutils.fs.ls(f"{base}/ingestion_date={ingestion_date}")
            return [f.path for f in files if f.path.endswith(".json")]
        except Exception:
            return []
    else:
        all_files = []
        for p in dbutils.fs.ls(base):
            try:
                all_files.extend([f.path for f in dbutils.fs.ls(p.path) if f.path.endswith(".json")])
            except Exception:
                pass
        return all_files


def folder_exists_dbfs(path):
    try:
        dbutils.fs.ls(path)
        return True
    except Exception:
        return False


def write_quarantine(df, entity_name, reason):
    """Save rejected rows to quarantine partition tagged with rejection reason."""
    if df.rdd.isEmpty():
        return
    (
        df.withColumn("reject_reason", F.lit(reason))
          .withColumn("reject_ts",     F.lit(RUN_TS).cast("timestamp"))
          .write.format("delta")
          .mode("append")
          .option("mergeSchema", "true")
          .save(f"{QUARANTINE_BASE}/{entity_name}")
    )


print("Helpers defined")

In [ ]:
# Cell 6 — Core transformation function (with full data quality pipeline)

# Columns that must never be negative (entity -> list of numeric cols to check)
NEGATIVE_CHECK_COLS = {
    "payments"     : ["amount_aud"],
    "sessions"     : ["energy_kwh", "duration_minutes"],
    "fleet"        : ["battery_capacity", "range_km"],
    "chargers"     : ["power_kw"],
    "vehicles"     : ["battery_capacity", "range_km"],
    "stations"     : ["total_chargers"],
    "energy_prices": ["price_per_kwh"],
    "tariffs"      : ["price_per_kwh", "price_per_min"],
    "cities"       : ["population"],
    "weather"      : ["humidity_pct"],
}

def transform_entity(entity_name, config, load_type, ingestion_date):
    result = {
        "entity_name"       : entity_name,
        "status"            : "failed",
        "bronze_rows"       : 0,
        "null_pk_dropped"   : 0,
        "null_cdc_dropped"  : 0,
        "corrupt_quarantine": 0,
        "negative_quarantine": 0,
        "duplicate_dropped" : 0,
        "silver_rows"       : 0,
        "error"             : None,
    }

    try:
        natural_key = config["natural_key"]
        cdc_field   = config["cdc_field"]
        cast_map    = config["cast"]
        silver_path = f"{SILVER_API_BASE}/{entity_name}"

        # ── Step 1: Discover Bronze JSON files ────────────────────────────────
        paths = get_bronze_paths(entity_name, ingestion_date)
        if not paths:
            result["status"] = "skipped"
            result["error"]  = "No Bronze JSON files found"
            return result

        # ── Step 2: Read raw JSON ─────────────────────────────────────────────
        raw_df = spark.read.option("multiline", "true").json(paths)

        if "data" not in raw_df.columns:
            result["error"] = "Column 'data' not found — schema mismatch"
            return result

        # ── Step 3: Explode data[] — handle both struct and string array types
        exploded_df = raw_df.select(F.explode(F.col("data")).alias("r"))
        if dict(exploded_df.dtypes)["r"] == "string":
            # data[] came back as array<string> — parse each element as JSON
            flat_df = (
                exploded_df
                .select(F.from_json(F.col("r"), MapType(StringType(), StringType())).alias("r"))
                .select("r.*")
            )
        else:
            flat_df = exploded_df.select("r.*")

        result["bronze_rows"] = flat_df.count()

        # ── Step 4: Trim whitespace on all string columns ─────────────────────
        # Catches values like "  pending  " that would fail status comparisons
        string_cols = [c for c, t in flat_df.dtypes if t == "string"]
        for col in string_cols:
            flat_df = flat_df.withColumn(col, F.trim(F.col(col)))

        # ── Step 5: Drop rows with NULL primary key ───────────────────────────
        # A record with no natural key cannot be identified or merged — reject it
        null_pk_df = flat_df.filter(F.col(natural_key).isNull() | (F.col(natural_key) == ""))
        result["null_pk_dropped"] = null_pk_df.count()
        if result["null_pk_dropped"] > 0:
            write_quarantine(null_pk_df, entity_name, f"null_primary_key:{natural_key}")
        flat_df = flat_df.filter(F.col(natural_key).isNotNull() & (F.col(natural_key) != ""))

        # ── Step 6: Drop rows with NULL CDC timestamp ─────────────────────────
        # updated_at drives dedup and watermark — a row without it is unusable
        null_cdc_df = flat_df.filter(F.col(cdc_field).isNull() | (F.col(cdc_field) == ""))
        result["null_cdc_dropped"] = null_cdc_df.count()
        if result["null_cdc_dropped"] > 0:
            write_quarantine(null_cdc_df, entity_name, f"null_cdc_field:{cdc_field}")
        flat_df = flat_df.filter(F.col(cdc_field).isNotNull() & (F.col(cdc_field) != ""))

        # ── Step 7: Cast columns to correct types ─────────────────────────────
        # PySpark returns NULL (not error) when a cast fails — e.g. "abc" cast to decimal = NULL
        # We detect those NULLs in numeric cols that had non-null values before cast = corrupt rows
        cast_exprs = []
        for col_name, col_type in cast_map.items():
            if col_name in flat_df.columns:
                cast_exprs.append(F.col(col_name).cast(col_type).alias(col_name))
            else:
                cast_exprs.append(F.lit(None).cast(col_type).alias(col_name))

        registry_cols = set(cast_map.keys())
        extra_cols    = [F.col(c).cast("string").alias(c)
                         for c in flat_df.columns if c not in registry_cols]

        typed_df = flat_df.select(cast_exprs + extra_cols)

        # Detect corrupt rows: numeric cols that are now NULL but were non-null strings before cast
        numeric_types = ("decimal", "integer", "long", "double", "float")
        numeric_cols  = [c for c, t in cast_map.items()
                         if any(t.startswith(nt) for nt in numeric_types)
                         and c in flat_df.columns and c != natural_key and c != cdc_field]

        if numeric_cols:
            # A row is corrupt if any numeric col is NULL after cast but was NOT null before
            corrupt_condition = F.lit(False)
            for c in numeric_cols:
                was_not_null = flat_df.filter(F.col(c).isNotNull()).count() > 0
                if was_not_null:
                    corrupt_condition = corrupt_condition | (
                        typed_df[c].isNull() & flat_df[c].isNotNull()
                    )
            # Re-join pre/post cast to find corrupted rows — simpler: flag nulls in typed_df
            null_after_cast = F.lit(False)
            for c in numeric_cols:
                null_after_cast = null_after_cast | typed_df[c].isNull()

            corrupt_df = typed_df.filter(null_after_cast)
            result["corrupt_quarantine"] = corrupt_df.count()
            if result["corrupt_quarantine"] > 0:
                write_quarantine(corrupt_df, entity_name, "corrupt_cast_to_null")
            typed_df = typed_df.filter(~null_after_cast)

        # ── Step 8: Quarantine rows with negative numeric values ──────────────
        # e.g. amount=-50, energy_kwh=-1 are invalid in this domain
        neg_cols = NEGATIVE_CHECK_COLS.get(entity_name, [])
        neg_cols = [c for c in neg_cols if c in typed_df.columns]

        if neg_cols:
            negative_condition = F.lit(False)
            for c in neg_cols:
                negative_condition = negative_condition | (F.col(c) < 0)

            negative_df = typed_df.filter(negative_condition)
            result["negative_quarantine"] = negative_df.count()
            if result["negative_quarantine"] > 0:
                write_quarantine(negative_df, entity_name, "negative_numeric_value")
            typed_df = typed_df.filter(~negative_condition)

        # ── Step 9: Add Silver audit columns ──────────────────────────────────
        typed_df = (
            typed_df
            .withColumn("silver_ingested_at", F.lit(RUN_TS).cast("timestamp"))
            .withColumn("silver_load_type",   F.lit(load_type))
            .withColumn("silver_pipeline",    F.lit(PIPELINE_NAME))
        )

        # ── Step 10: Deduplicate on natural key (keep latest updated_at) ──────
        # Handles same record appearing across multiple Bronze pages or ingestion dates
        window = Window.partitionBy(natural_key).orderBy(F.col(cdc_field).desc())
        before_dedup = typed_df.count()
        deduped_df = (
            typed_df
            .withColumn("_row_num", F.row_number().over(window))
            .filter(F.col("_row_num") == 1)
            .drop("_row_num")
        )
        result["duplicate_dropped"] = before_dedup - deduped_df.count()

        # ── Step 11: Write to Silver Delta ────────────────────────────────────
        writer = (
            deduped_df.write
            .format("delta")
            .option("mergeSchema", "true")
        )

        if load_type == "full" or not folder_exists_dbfs(silver_path):
            writer.mode("overwrite").save(silver_path)
        else:
            if DeltaTable.isDeltaTable(spark, silver_path):
                delta_table = DeltaTable.forPath(spark, silver_path)
                (
                    delta_table.alias("target")
                    .merge(
                        deduped_df.alias("source"),
                        f"target.{natural_key} = source.{natural_key}"
                    )
                    .whenMatchedUpdateAll()
                    .whenNotMatchedInsertAll()
                    .execute()
                )
            else:
                writer.mode("overwrite").save(silver_path)

        result["silver_rows"] = deduped_df.count()
        result["status"]      = "succeeded"

    except Exception as e:
        result["error"] = str(e)

    return result


print("transform_entity() defined with full data quality pipeline")

In [ ]:
# Cell 7 — Run transformation for all 17 entities sequentially
# (PySpark already parallelises reads/writes internally — sequential Python loop is fine)

print(f"Starting Silver transformation — load_type={LOAD_TYPE}")
print("-" * 60)

results = []
for entity_name, config in ENTITY_REGISTRY.items():
    date_filter = INGESTION_DATE if LOAD_TYPE == "incremental" else ""
    print(f"  Processing: {entity_name} ...", end=" ")
    r = transform_entity(entity_name, config, LOAD_TYPE, date_filter)
    results.append(r)
    if r["status"] == "succeeded":
        print(f"OK  ({r['bronze_rows']} bronze rows -> {r['silver_rows']} silver rows)")
    elif r["status"] == "skipped":
        print(f"SKIP  ({r['error']})")
    else:
        print(f"FAILED  ({r['error']})")

print("-" * 60)
print("All entities processed")

In [ ]:
# Cell 8 — Run summary with data quality breakdown

succeeded = [r for r in results if r["status"] == "succeeded"]
skipped   = [r for r in results if r["status"] == "skipped"]
failed    = [r for r in results if r["status"] == "failed"]

print("=" * 90)
print("SILVER API TRANSFORMATION v3 — RUN SUMMARY")
print("=" * 90)
print(f"  load_type      : {LOAD_TYPE}")
print(f"  ingestion_date : {INGESTION_DATE or '(all partitions)'}")
print(f"  run_timestamp  : {RUN_TS}")
print(f"  entities total : {len(results)}")
print(f"  succeeded      : {len(succeeded)}")
print(f"  skipped        : {len(skipped)}")
print(f"  failed         : {len(failed)}")
print()
print(f"  {'ENTITY':<25} {'STATUS':<10} {'BRONZE':>7} {'NULL_PK':>8} {'NULL_CDC':>9} {'CORRUPT':>8} {'NEGATIVE':>9} {'DUPS':>6} {'SILVER':>7}")
print(f"  {'-'*25} {'-'*10} {'-'*7} {'-'*8} {'-'*9} {'-'*8} {'-'*9} {'-'*6} {'-'*7}")

for r in results:
    if r["status"] == "succeeded":
        tag = "[OK]  "
    elif r["status"] == "skipped":
        tag = "[SKIP]"
    else:
        tag = "[FAIL]"
    print(
        f"  {tag} {r['entity_name']:<25} {r['status']:<10}"
        f" {r['bronze_rows']:>7}"
        f" {r['null_pk_dropped']:>8}"
        f" {r['null_cdc_dropped']:>9}"
        f" {r['corrupt_quarantine']:>8}"
        f" {r['negative_quarantine']:>9}"
        f" {r['duplicate_dropped']:>6}"
        f" {r['silver_rows']:>7}"
    )
    if r["status"] == "failed":
        print(f"         Error: {r['error']}")

print()
total_quarantined = sum(
    r["null_pk_dropped"] + r["null_cdc_dropped"] + r["corrupt_quarantine"] + r["negative_quarantine"]
    for r in results
)
total_silver = sum(r["silver_rows"] for r in results)
print(f"  Total rows quarantined : {total_quarantined}")
print(f"  Total rows in Silver   : {total_silver}")
print(f"  Quarantine path        : {QUARANTINE_BASE}/<entity>/")
print()

if failed:
    raise Exception(f"{len(failed)} entity(ies) failed — check output above.")
else:
    print(f"Silver transformation complete. {len(succeeded)} entities written successfully.")

In [ ]:
# Cell 9 — Verify Silver + Quarantine

print("SILVER VERIFICATION — spot check")
print("-" * 50)
for entity in ["payments", "sessions", "customers"]:
    silver_path = f"{SILVER_API_BASE}/{entity}"
    if not folder_exists_dbfs(silver_path):
        print(f"  {entity}: NOT FOUND")
        continue
    df = spark.read.format("delta").load(silver_path)
    print(f"  {entity:<25} rows={df.count():>6}  cols={len(df.columns)}")
    if entity == "payments":
        df.printSchema()
        df.show(3, truncate=True)

print()
print("QUARANTINE CHECK — rows rejected by data quality rules")
print("-" * 50)
for entity in ["payments", "sessions", "customers"]:
    q_path = f"{QUARANTINE_BASE}/{entity}"
    if not folder_exists_dbfs(q_path):
        print(f"  {entity:<25} no quarantine rows (clean data)")
        continue
    q_df = spark.read.format("delta").load(q_path)
    print(f"  {entity:<25} quarantine rows={q_df.count()}")
    q_df.groupBy("reject_reason").count().show(truncate=False)